# Rental Property Lookup: RAG Baseline

A minimal RAG baseline using minsearch, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

The user enters a rental property address. We retrieve the matching building from public NYC records and ask the LLM to turn it into a structured record: property type, owner, owner type, property manager, management software, and unit count, each with evidence and a confidence level.

In [ ]:
import json
import re
from pathlib import Path

import requests
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

We use NYC open data (no API key needed) as a stand-in for county assessor and landlord registration records:

- [HPD Registrations](https://data.cityofnewyork.us/d/tesw-yqqr) and [Registration Contacts](https://data.cityofnewyork.us/d/feu5-w2e2): every NYC rental building with 3+ units must register its owner, managing agent, and site manager each year
- [HPD Buildings](https://data.cityofnewyork.us/d/kj4p-ruqc): legal unit counts
- [PLUTO](https://data.cityofnewyork.us/d/64uk-42ks): tax lot data with the owner of record and building class

We pull one zip code (11226, Flatbush) and join everything into one document per building. The result is cached in `data/` so the API is only called once.

In [ ]:
DATA_DIR = Path('../data')
ZIP_CODE = '11226'  # Flatbush, Brooklyn - dense rental stock, lots of LLC owners
DATA_FILE = DATA_DIR / f'nyc_rental_buildings_{ZIP_CODE}.json'

SOCRATA = 'https://data.cityofnewyork.us/resource'
HPD_REGISTRATIONS = f'{SOCRATA}/tesw-yqqr.json'  # one row per registered building
HPD_CONTACTS = f'{SOCRATA}/feu5-w2e2.json'       # owners, agents, site managers per registration
HPD_BUILDINGS = f'{SOCRATA}/kj4p-ruqc.json'      # legal unit counts
PLUTO = f'{SOCRATA}/64uk-42ks.json'              # tax lot data: owner of record, building class


def socrata_get(url, **params):
    params.setdefault('$limit', 50000)
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


def fetch_contacts(registration_ids, batch_size=200):
    contacts = []
    for i in range(0, len(registration_ids), batch_size):
        batch = registration_ids[i:i + batch_size]
        id_list = ','.join(f"'{r}'" for r in batch)
        contacts.extend(socrata_get(HPD_CONTACTS, **{'$where': f'registrationid in ({id_list})'}))
    return contacts


def format_contact(c):
    name = c.get('corporationname') or ' '.join(
        p for p in [c.get('firstname'), c.get('lastname')] if p
    )
    business_address = ' '.join(
        p for p in [c.get('businesshousenumber'), c.get('businessstreetname'),
                    c.get('businesscity'), c.get('businessstate')] if p
    )
    if business_address:
        return f'{name} ({business_address})'
    return name


def fetch_nyc_rental_buildings(zip_code):
    registrations = socrata_get(HPD_REGISTRATIONS, zip=zip_code)
    buildings = socrata_get(HPD_BUILDINGS, zip=zip_code)
    lots = socrata_get(PLUTO, zipcode=zip_code)
    contacts = fetch_contacts([r['registrationid'] for r in registrations])

    buildings_by_id = {b['buildingid']: b for b in buildings}
    lots_by_block_lot = {(l['block'], l['lot']): l for l in lots}

    contacts_by_registration = {}
    for c in contacts:
        contacts_by_registration.setdefault(c['registrationid'], []).append(c)

    documents = []
    for r in registrations:
        building = buildings_by_id.get(r['buildingid'], {})
        lot = lots_by_block_lot.get((r['block'], r['lot']), {})

        roles = {}
        for c in contacts_by_registration.get(r['registrationid'], []):
            roles.setdefault(c['type'], []).append(format_contact(c))

        def role(*types):
            return '; '.join(sorted({n for t in types for n in roles.get(t, [])}))

        units = int(building.get('legalclassa') or 0) + int(building.get('legalclassb') or 0)

        documents.append({
            'address': f"{r.get('housenumber', '')} {r.get('streetname', '')}, {r['boro']}, NY {r['zip']}",
            'owner': role('CorporateOwner', 'IndividualOwner', 'JointOwner'),
            'head_officer': role('HeadOfficer', 'Officer'),
            'managing_agent': role('Agent'),
            'site_manager': role('SiteManager'),
            'assessor_owner': lot.get('ownername', ''),
            'building_class': lot.get('bldgclass', ''),
            'year_built': lot.get('yearbuilt', ''),
            'units': units or int(lot.get('unitsres') or 0),
            'zip': r['zip'],
            'bbl': lot.get('bbl', '').split('.')[0],
            'hpd_building_id': r['buildingid'],
            'hpd_registration_id': r['registrationid'],
            'last_registration_date': r.get('lastregistrationdate', '')[:10],
        })

    return documents


if DATA_FILE.exists():
    documents = json.loads(DATA_FILE.read_text())
else:
    documents = fetch_nyc_rental_buildings(ZIP_CODE)
    DATA_FILE.write_text(json.dumps(documents, indent=2))

len(documents)

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify. Users mostly search by address, but they can also search by owner or manager name.

In [ ]:
index = Index(
    text_fields=["address", "owner", "head_officer", "managing_agent", "site_manager", "assessor_owner"],
    keyword_fields=["zip", "building_class"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [ ]:
STREET_ABBREVIATIONS = {
    'st': 'street', 'ave': 'avenue', 'av': 'avenue', 'blvd': 'boulevard',
    'rd': 'road', 'pl': 'place', 'pkwy': 'parkway', 'e': 'east', 'w': 'west',
}


def normalize_address(query):
    # HPD writes "EAST 15 STREET"; users write "E 15th St"
    query = re.sub(r'\b(\d+)(st|nd|rd|th)\b', r'\1', query, flags=re.IGNORECASE)
    return re.sub(
        r'\b[A-Za-z]+\b',
        lambda m: STREET_ABBREVIATIONS.get(m.group().lower(), m.group()),
        query,
    )


def search(query):
    return index.search(
        normalize_address(query),
        boost_dict={'address': 3},
        num_results=5,
    )

In [ ]:
instructions = """
You're a research assistant for a team that models the US residential rental
market. The user gives you a rental property address. Find the building in the
CONTEXT that matches the address (house number and street must match; if none
does, say the property wasn't found and stop) and return a structured record
with these fields:

- Property type: single-family, multifamily, townhome, mixed-use, etc.
  Derive it from the NYC building class (A = one-family, B = two-family,
  C = walk-up apartments, D = elevator apartments, R = condo, S = mixed
  residential and retail) and the unit count.
- Owner of record: the registered owner entity exactly as filed.
- Normalized owner: the parent company or person behind the owner entity.
  Single-purpose LLCs often share a business address or head officer with
  their parent; use those links, and say so when you do.
- Owner type: institutional, mid-size, or individual.
- Property manager: the managing agent, or the site manager if no agent
  is listed.
- Property management software: the public records don't contain this.
  Report "unknown" and say what web research would resolve it (for example,
  finding the property's resident portal).
- Number of units.

Where each CONTEXT field comes from:
- owner, head_officer, managing_agent, site_manager: HPD Registration Contacts
- assessor_owner, building_class, year_built, bbl: PLUTO (tax lot records)
- units: HPD Buildings (legal unit count)

For every field give its own evidence (the dataset and field it came from,
plus the HPD registration ID or BBL) and its own confidence level (high,
medium, or low). An LLC alone doesn't make an owner institutional.
Use only the CONTEXT. Never guess a value; mark it unknown instead. Flag
conflicts between sources, such as an HPD owner that differs from the PLUTO
owner of record.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [ ]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [ ]:
print(rag("Who owns and manages 123 Linden Blvd, Brooklyn, NY 11226?"))